# Day 12 — Specialization D: Recommender Systems
Matrix factorization (implicit-style ALS via `implicit`) on MovieLens-100k.

Install: `pip install implicit pandas scipy requests`


In [ ]:
import io, zipfile, requests, pandas as pd, numpy as np
from pathlib import Path

ROOT = Path('ml-100k')
if not ROOT.exists():
    r = requests.get('https://files.grouplens.org/datasets/movielens/ml-100k.zip', timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall('.')
ratings = pd.read_csv(ROOT / 'u.data', sep='\t', names=['user', 'item', 'rating', 'ts'])
items   = pd.read_csv(ROOT / 'u.item', sep='|', encoding='latin-1', header=None,
                      names=['item', 'title'] + list(range(22)), usecols=['item', 'title'])
print(ratings.shape); ratings.head()


In [ ]:
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

n_users = ratings['user'].max() + 1
n_items = ratings['item'].max() + 1
ui = csr_matrix((ratings['rating'].astype('float32'), (ratings['user'], ratings['item'])), shape=(n_users, n_items))

model = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20)
model.fit(ui)


In [ ]:
title_of = items.set_index('item')['title'].to_dict()

def recommend(user_id, k=10):
    ids, scores = model.recommend(user_id, ui[user_id], N=k, filter_already_liked_items=True)
    return [(title_of.get(int(i), str(int(i))), float(s)) for i, s in zip(ids, scores)]

for u in [1, 50, 196]:
    print(f'\nTop picks for user {u}:')
    for t, s in recommend(u, 5):
        print(f'  {s:6.2f}  {t}')


In [ ]:
# Find similar items
def similar(item_title, k=10):
    item_id = items[items['title'].str.contains(item_title, case=False)].iloc[0]['item']
    ids, scores = model.similar_items(int(item_id), N=k)
    return [(title_of.get(int(i), str(int(i))), float(s)) for i, s in zip(ids, scores)]

similar('Star Wars')


### Exercises
- Hold out 20% of ratings and compute Recall@10 / NDCG@10.
- Build a content-based recommender using `u.item` genre flags.
- Try a two-tower neural recsys with PyTorch.
